# router_final — training, gate post-hoc, deployment A3, evaluasi (Kaggle T4)

Satu seed per sesi Kaggle (batas 12 jam). Urutan per seed: training → cek checkpoint → gate → ekspor → evaluasi →
suite final (H1–H7, latensi ujung-ke-ujung) → sensitivitas baseline `best.pt` → ablasi (seed 42 dan VisDrone saja).
Kode = folder `router_final/` di branch `final-arsitektur`, di atas **Ultralytics stok 8.3.240** (bukan fork di root repo).

**Melanjutkan di sesi baru:** unggah `arsip_<model>_seed<seed>.zip` sebagai dataset Kaggle, isi `RESUME_FROM`, lalu matikan
langkah yang sudah selesai di `STEPS`. Arsip disalin ke `/kaggle/working` supaya semua langkah bisa menulis ke `RUN`.
Engine TensorRT tidak ikut diarsip; langkah `eval` (sel 6) butuh `export` di sesi yang sama.

In [ ]:
# ===== 0. Kode dipin =====
REPO_URL = "https://github.com/maugi1547/thesis-Difficulty-Aware-Router"
CODE_COMMIT = "ISI_HASH_COMMIT"          # hash commit branch final-arsitektur yang dipakai (wajib, bukan nama branch)
SEED = 42                                # satu seed per sesi: 42, 7, 2026
CONFIG = "configs/kitti.yaml"
RUN_REPRO_CHECK = True                   # 2 × 2 epoch, bandingkan bobot (±30 menit). Wajib sekali per lingkungan baru.
# Langkah per sesi. Sesi lanjutan: RESUME_FROM = folder arsip di /kaggle/input, dan set langkah yang sudah selesai = False.
STEPS = dict(train=True, gate=True, export=True, eval=True, suite=True,
             sens_best=True,                 # sensitivitas: suite ulang dengan baseline best.pt (±30 menit)
             ablation=SEED == 42)            # ablasi + uji berpasangan (±40 menit): seed 42 KITTI + VisDrone
RESUME_FROM = None                       # mis. "/kaggle/input/arsip-kitti-seed7" (berisi weights/last.pt); None = training di sesi ini
# Baseline stok, TIDAK dilatih ulang. last.pt = aturan sama dengan router (Tahap 4/4B lama memakai best.pt;
# angka baseline tidak sebanding dengan tabel lama). Sensitivitas best.pt otomatis di sel 7b (STEPS["sens_best"]).
BASELINES = {
    "configs/kitti.yaml": dict(
        vanilla="/kaggle/input/models/agungmaugi/yolov8n-kitti/pytorch/default/1/Skenario1_YOLOv8n_Standar/weights/last.pt",
        p2static="/kaggle/input/models/agungmaugi/yolov8n-p2-statis-kitti/pytorch/default/1/Skenario2_YOLOv8n_P2_Statis/weights/last.pt"),
    "configs/visdrone.yaml": dict(
        vanilla="/kaggle/input/models/agungmaugi/yolov8n-visdrone/pytorch/default/1/content/thesis-Difficulty-Aware-Router/Tesis_VisDrone/Skenario1_YOLOv8n_Standar/weights/last.pt",
        p2static="/kaggle/input/models/agungmaugi/yolov8n-p2-visdrone/pytorch/default/1/Skenario2_YOLOv8n_P2_Statis/weights/last.pt"),
}
assert CODE_COMMIT != "ISI_HASH_COMMIT", "Isi CODE_COMMIT"
assert not (STEPS["train"] and RESUME_FROM), "RESUME_FROM diisi -> STEPS['train'] harus False"

import os, subprocess
REPO = "/kaggle/working/thesis-Difficulty-Aware-Router"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q", "--all"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", CODE_COMMIT], check=True)
RF = f"{REPO}/router_final"
os.chdir(RF)                              # PENTING: jangan di root repo (folder ultralytics/ fork akan menutupi paket pip)
!pip install -q --no-deps ultralytics==8.3.240
!pip install -q onnx onnxscript onnxruntime pycocotools tensorrt
print(subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True))

In [ ]:
# ===== 1. Uji unit (CPU, <1 menit): ultralytics stok, graf, loss, gate, B2, ONNX A3 =====
!python -m pytest -q -p no:cacheprovider tests

In [ ]:
# ===== 2. Uji reproduksi GPU: resep sama dua kali -> bobot harus identik bit per bit =====
if RUN_REPRO_CHECK:
    !python scripts/check_reproducibility.py --config {CONFIG} --seed {SEED} --override epochs=2 project=/kaggle/working/repro

In [ ]:
# ===== 3. Training detektor (100 epoch) atau lanjut dari arsip =====
import yaml, glob, shutil, pathlib
cfg = yaml.safe_load(open(CONFIG))
RUN = f"{cfg['train']['project']}/{cfg['name']}_seed{SEED}"
if STEPS["train"]:
    !python scripts/train.py --config {CONFIG} --seed {SEED}
    # last.pt vs best.pt (best = fitness Branch A saja) di VAL, A dan B + bootstrap; laporan saja, last.pt tetap dipakai
    !python scripts/check_checkpoint.py --config {CONFIG} --run {RUN} --out {RUN}/cek_checkpoint
else:
    src = [pathlib.Path(p).parent.parent for p in glob.glob(f"{RESUME_FROM}/**/weights/last.pt", recursive=True)]
    assert len(src) == 1, f"arsip harus berisi tepat satu weights/last.pt: {src}"
    shutil.copytree(src[0], RUN, dirs_exist_ok=True)              # /kaggle/input read-only -> salin ke working
    print("lanjut dari", src[0])
print(RUN); print(open(f"{RUN}/state_hash.json").read())
assert f"seed{SEED}" in RUN and str(cfg["name"]) in RUN

In [ ]:
# ===== 4. Gate post-hoc (train) + thr0 (val) =====
if STEPS["gate"]:
    !python scripts/fit_gate.py --config {CONFIG} --weights {RUN}/weights/last.pt --out {RUN}/gate

In [ ]:
# ===== 5. Ekspor A3 (ONNX If FP16) + verifikasi ORT + engine TensorRT =====
if STEPS["export"]:
    !python scripts/export.py --config {CONFIG} --weights {RUN}/weights/last.pt --gate {RUN}/gate/gate.json --out {RUN}/deploy --trt

In [ ]:
# ===== 6. Evaluasi test: TensorRT (sistem deploy) + PyTorch FP32 (pemeriksaan silang) =====
if STEPS["eval"]:
    !python scripts/evaluate.py --config {CONFIG} --weights {RUN}/weights/last.pt --gate {RUN}/gate/gate.json --out {RUN}/eval_test_trt --backend trt --engine {RUN}/deploy/a3_fp16.engine
    !python scripts/evaluate.py --config {CONFIG} --weights {RUN}/weights/last.pt --gate {RUN}/gate/gate.json --out {RUN}/eval_test_pytorch --backend pytorch

In [ ]:
# ===== 7. Evaluasi final terkunci + latensi ujung-ke-ujung (H1–H7, bootstrap, baseline, heuristik, kaskade) =====
# ±30–90 menit T4: cache train/val/test + 2 build engine + pass akurasi + latensi komponen + ujung-ke-ujung + bootstrap
B = BASELINES[CONFIG]
if STEPS["suite"]:
    !python scripts/suite.py run --config {CONFIG} --seed {SEED} --weights {RUN}/weights/last.pt --gate {RUN}/gate/gate.json \
        --vanilla "{B['vanilla']}" --p2static "{B['p2static']}" --out /kaggle/working/suite --zip /kaggle/working

In [ ]:
# ===== 7b. Sensitivitas: baseline best.pt (router, gate, dan engine router tetap) =====
# Tanpa --reuse: cache intermediate berisi deteksi baseline last.pt. Nama terpisah -> tidak tercampur saat merge.
BB = {k: v.replace("/weights/last.pt", "/weights/best.pt") for k, v in BASELINES[CONFIG].items()}
NAME_BB = f"{cfg['name']}_seed{SEED}_baseline_best"
if STEPS["sens_best"]:
    assert all(os.path.exists(p) for p in BB.values()), BB
    !python scripts/suite.py run --config {CONFIG} --seed {SEED} --weights {RUN}/weights/last.pt --gate {RUN}/gate/gate.json \
        --vanilla "{BB['vanilla']}" --p2static "{BB['p2static']}" --name {NAME_BB} --out /kaggle/working/suite --zip /kaggle/working

In [ ]:
# ===== 8. Ablasi + uji berpasangan (G1 target, G3 fitur, G4 classifier, G5 label, B anggaran) =====
# Menulis RUN/ablasi/{ablasi.csv, ablasi_uji.csv, cache_ablasi.pkl}; analisis ulang tanpa GPU: tambah --reuse-cache
if STEPS["ablation"]:
    !python scripts/ablate.py --config {CONFIG} --weights {RUN}/weights/last.pt --out {RUN}/ablasi

In [ ]:
# ===== 9. Arsip (tanpa engine & ONNX besar) — unduh ketiganya: arsip + suite_* (utama & baseline_best) =====
keep = pathlib.Path(f"/kaggle/working/arsip_{cfg['name']}_seed{SEED}")
shutil.copytree(RUN, keep, ignore=shutil.ignore_patterns("*.engine", "*.onnx", "epoch*.pt"), dirs_exist_ok=True)
shutil.make_archive(str(keep), "zip", keep)
print(keep.with_suffix(".zip"))
print(sorted(str(p) for p in pathlib.Path("/kaggle/working").glob("suite_*.zip")))   # hasil + intermediate evaluasi final

## Ringkasan gabungan (sesi terakhir; CPU cukup)
Unggah semua `suite_<model>.zip` dan `suite_<model>_baseline_best.zip` (bukan `_intermediate`) sebagai satu dataset Kaggle,
lalu jalankan sel ini saja (setelah sel 0). Hasil utama dan sensitivitas digabung terpisah: model dipilih lewat namanya,
jadi folder `_baseline_best` tidak ikut pooling utama.

In [ ]:
# ===== 10. Gabung: utama (baseline last.pt) dan sensitivitas (baseline best.pt) — tabel kriteria, pooling KITTI 3 seed, H3 + Holm, Pareto =====
import shutil
MERGE_GLOB = "/kaggle/input/**/suite_*"     # sesuaikan dengan nama dataset hasil
MODELS = ["kitti_seed42", "kitti_seed7", "kitti_seed2026", "visdrone_seed42"]
MODELS_BB = " ".join(m + "_baseline_best" for m in MODELS)
MODELS = " ".join(MODELS)
!python scripts/suite.py merge --out /kaggle/working/suite_gabung --glob "{MERGE_GLOB}" --models {MODELS}
!python scripts/suite.py merge --out /kaggle/working/suite_gabung_baseline_best --glob "{MERGE_GLOB}" --models {MODELS_BB}
for d in ("suite_gabung", "suite_gabung_baseline_best"):
    print(shutil.make_archive(f"/kaggle/working/{d}", "zip", f"/kaggle/working/{d}"))